[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_methods/04_polynomial_and_spline_interpolation/exercises.ipynb)

# Exercises — Topic 04: Polynomial and Spline Interpolation

20 fully solved problems in 4 levels: concept checks, foundational computations and derivations, AI/ML and physics applications, and challenge proofs.

## Level 0 — Concept Check

### Problem L0.1: One polynomial, many faces

For the data $(0, 1), (1, 3), (2, 7)$, build the interpolant in both the Lagrange and the Newton divided-difference form, expand both, and explain why they must agree.

**Solution.**

**Lagrange form.** With nodes $0, 1, 2$:

$$
L_0 = \frac{(x-1)(x-2)}{(0-1)(0-2)} = \frac{x^2 - 3x + 2}{2}, \quad L_1 = \frac{x(x-2)}{(1)(-1)} = -x^2 + 2x, \quad L_2 = \frac{x(x-1)}{(2)(1)} = \frac{x^2 - x}{2}.
$$

$$
p(x) = 1 \cdot \frac{x^2 - 3x + 2}{2} + 3(-x^2 + 2x) + 7 \cdot \frac{x^2 - x}{2} = \left(\tfrac12 - 3 + \tfrac72\right)x^2 + \left(-\tfrac32 + 6 - \tfrac72\right)x + 1 = x^2 + x + 1 .
$$

**Newton form.** Divided-difference table:

$$
f[0,1] = \frac{3-1}{1} = 2, \quad f[1,2] = \frac{7-3}{1} = 4, \quad f[0,1,2] = \frac{4-2}{2-0} = 1 .
$$

$$
p(x) = 1 + 2(x - 0) + 1 \cdot (x-0)(x-1) = 1 + 2x + x^2 - x = x^2 + x + 1 .
$$

**Why they agree.** Both are polynomials of degree $\le 2$ matching three values at three distinct nodes. By uniqueness (a nonzero degree-$\le 2$ difference cannot have 3 roots), they are the same polynomial.

$$
\boxed{p(x) = x^2 + x + 1 \text{ in every basis; the forms differ only as algorithms}}
$$

*Key takeaway:* Lagrange, Newton, monomial, and barycentric are four **representations** of one unique object — choose by cost and stability, not by mathematics.

### Problem L0.2: Why more points can be worse

Explain in first-principles terms why interpolating $f(x) = \frac{1}{1+25x^2}$ at $n+1$ equispaced points on $[-1,1]$ gets *worse* as $n$ grows, even though $f$ is infinitely differentiable, and state the two independent cures.

**Solution.**

The error theorem gives $f(x) - p_n(x) = \frac{f^{(n+1)}(\xi_x)}{(n+1)!}\omega_{n+1}(x)$. Both factors misbehave:

1. **The derivative factor.** $f$ has poles at $x = \pm i/5$ in the complex plane. Cauchy's estimate forces $\lVert f^{(n+1)} \rVert_\infty$ to grow like $(n+1)!\,5^{\,n+1}$ — *faster* than $(n+1)!$, so the factorial denominator does not tame it. Smoothness on the real line is not enough; what matters is the distance to the nearest complex singularity.

2. **The node factor.** For equispaced nodes $\omega_{n+1}$ is exponentially larger near the endpoints than in the middle — the ratio of peak to central value grows like $2^{n}$. So the interpolant develops wild oscillations near $x = \pm 1$.

Measured numerically on a fine grid, $\lVert f - p_n \rVert_\infty$ for equispaced nodes is $1.92$ at $n = 10$, $59.8$ at $n = 20$, and $2388$ at $n = 30$ — geometric divergence.

**Cure 1 — move the nodes.** Chebyshev points make $\omega_{n+1}$ equioscillate. Same $f$, same degrees: errors $0.132$, $0.0177$, $0.00243$ — geometric *convergence*.

**Cure 2 — stop raising the degree.** A cubic spline on the same equispaced nodes gives $0.0220$, $0.00318$, $0.000824$, converging like $O(h^4)$ with no oscillation at all.

$$
\boxed{\text{Runge divergence comes from equispaced nodes, not from smoothness; fix the nodes or lower the degree}}
$$

*Key takeaway:* Interpolation error is a product of an uncontrollable derivative factor and a fully controllable node factor — always attack the node factor.

### Problem L0.3: Counting degrees of freedom for a cubic spline

A cubic spline interpolates $n+1$ data points, so it has $n$ intervals. Count its unknowns and its constraints, show that exactly two conditions are missing, and describe the three standard ways of supplying them.

**Solution.**

**Unknowns.** Each of the $n$ intervals carries a cubic with 4 coefficients: $4n$ unknowns.

**Constraints.**
- Interpolation at the left and right end of every interval: $2n$ conditions (this also enforces $C^0$ continuity).
- Continuity of $s'$ at the $n-1$ interior knots: $n - 1$.
- Continuity of $s''$ at the $n-1$ interior knots: $n - 1$.

Total: $2n + 2(n-1) = 4n - 2$. So the system is underdetermined by exactly

$$
4n - (4n - 2) = 2 \text{ conditions.}
$$

**The three standard closures.**
- **Natural**: $s''(x_0) = s''(x_n) = 0$. Physically the beam is unloaded past the ends; mathematically it is the minimum-energy spline, but accuracy near the boundary drops to $O(h^2)$.
- **Clamped (complete)**: $s'(x_0) = f'(a)$, $s'(x_n) = f'(b)$. Needs derivative data but preserves $O(h^4)$ throughout, with the sharp bound $\frac{5}{384}h^4 \lVert f^{(4)} \rVert_\infty$.
- **Not-a-knot**: require $s'''$ continuous at $x_1$ and $x_{n-1}$, i.e. the first two (and last two) pieces are a single cubic. Needs no extra data and keeps $O(h^4)$; it is SciPy's and MATLAB's default.

$$
\boxed{4n \text{ unknowns} - (4n-2) \text{ conditions} = 2 \text{ free end conditions}}
$$

*Key takeaway:* End conditions are not decoration — they are exactly the two missing equations, and the choice changes the boundary accuracy by two orders.

### Problem L0.4: Choosing an interpolation method

Pick a method for each situation and justify: (a) exact samples of a smooth function, and you may choose the sample locations; (b) exact samples on a fixed equispaced grid of 200 points; (c) noisy measurements with 5% scatter; (d) a probability-calibration map that must be nondecreasing and stay in $[0,1]$.

**Solution.**

(a) **Chebyshev nodes with barycentric evaluation.** You control the node factor $\omega_{n+1}$, so make it minimax-optimal; for $f$ analytic the error then decays geometrically, $O(\rho^{-n})$, and barycentric evaluation is $O(n)$ per point and backward stable.

(b) **Cubic spline (not-a-knot).** A degree-199 global polynomial on equispaced nodes would diverge catastrophically (Runge) and the Vandermonde conditioning would be $\sim 2.4^{200}$. The spline gives $O(h^4)$ with an $O(n)$ tridiagonal solve.

(c) **Do not interpolate — smooth.** Exact interpolation reproduces the noise and, worse, differentiating it amplifies the noise. Use a smoothing spline minimizing $\sum_i (y_i - g(x_i))^2 + \lambda \int (g'')^2$, with $\lambda$ chosen by generalized cross-validation.

(d) **PCHIP or isotonic regression.** An unconstrained cubic spline can overshoot: on the monotone data $(0,0),(1,0),(2,0),(3,1),(4,1),(5,1)$ the natural cubic spline reaches $1.109$ and dips to $-0.109$ — impossible values for a probability. PCHIP limits the slopes to preserve monotonicity, at the cost of $C^1$ instead of $C^2$.

$$
\boxed{\text{(a) Chebyshev + barycentric, (b) cubic spline, (c) smoothing spline, (d) PCHIP / isotonic}}
$$

*Key takeaway:* The method is chosen by three facts: who controls the nodes, whether the data are noisy, and what shape constraints the answer must satisfy.

## Level 1 — Foundation

### Problem L1.1: Divided differences and incremental extension

Build the Newton interpolant for $f$ sampled at $x = 0, 1, 2, 3$ with values $1, 2, 9, 28$. Then verify that adding the node $x = 4$ with $y = 65$ requires only $O(n)$ extra work, and identify $f$.

**Solution.**

**Divided-difference table.**

First order: $f[0,1] = \frac{2-1}{1} = 1$, $f[1,2] = \frac{9-2}{1} = 7$, $f[2,3] = \frac{28-9}{1} = 19$.

Second order: $f[0,1,2] = \frac{7-1}{2} = 3$, $f[1,2,3] = \frac{19-7}{2} = 6$.

Third order: $f[0,1,2,3] = \frac{6-3}{3} = 1$.

$$
p_3(x) = 1 + 1\cdot x + 3\,x(x-1) + 1\cdot x(x-1)(x-2) .
$$

Expanding: $1 + x + 3x^2 - 3x + x^3 - 3x^2 + 2x = x^3 + 1$. So $f(x) = x^3 + 1$.

**Adding $x_4 = 4$, $y_4 = 65$.** Only the *new diagonal* must be computed:

$$
f[3,4] = \frac{65 - 28}{1} = 37, \quad f[2,3,4] = \frac{37-19}{2} = 9, \quad f[1,2,3,4] = \frac{9-6}{3} = 1, \quad f[0,\ldots,4] = \frac{1-1}{4} = 0 .
$$

The top coefficient is $0$, so $p_4 = p_3$ — consistent with $f = x^3 + 1$ being exactly cubic (all divided differences of order $\ge 4$ vanish). Cost: $4$ new entries, i.e. $O(n)$, versus $O(n^2)$ to rebuild a Lagrange form.

$$
\boxed{p(x) = 1 + x + 3x(x-1) + x(x-1)(x-2) = x^3 + 1; \quad f[0,\ldots,4] = 0}
$$

*Key takeaway:* The Newton form is the *incremental* form — one new node costs one new diagonal — and a vanishing top divided difference certifies that the data are exactly polynomial.

### Problem L1.2: Table spacing for linear interpolation

A lookup table stores $\sin x$ on $[0, \pi/2]$ at equispaced points and uses linear interpolation between entries. How small must the spacing $h$ be to guarantee absolute error at most $10^{-6}$, and how many entries does that need? Then repeat for $10^{-8}$.

**Solution.**

**Error bound.** On one subinterval $[x_i, x_{i+1}]$ of length $h$, the error theorem with $n = 1$ gives

$$
f(x) - p_1(x) = \frac{f''(\xi)}{2}(x - x_i)(x - x_{i+1}).
$$

The quadratic $\lvert (x-x_i)(x-x_{i+1}) \rvert$ is maximized at the midpoint with value $h^2/4$, so

$$
\lVert f - p_1 \rVert_\infty \le \frac{h^2}{8}\,\lVert f'' \rVert_\infty .
$$

For $f = \sin$, $\lVert f'' \rVert_\infty = 1$.

**Tolerance $10^{-6}$.** $\frac{h^2}{8} \le 10^{-6} \iff h \le \sqrt{8 \times 10^{-6}} = 2.8284 \times 10^{-3}$. Number of intervals: $\frac{\pi/2}{2.8284\times10^{-3}} = 555.4$, so $556$ intervals and $\boxed{557}$ table entries.

**Tolerance $10^{-8}$.** $h \le \sqrt{8\times 10^{-8}} = 2.8284\times 10^{-4}$: $5554$ intervals, $5555$ entries — $10\times$ the memory for $100\times$ the accuracy, the signature $O(h^2)$ trade.

**Contrast with a cubic spline.** With $\lVert f^{(4)} \rVert_\infty = 1$, $\frac{5}{384}h^4 \le 10^{-8}$ gives $h \le (7.68\times10^{-7})^{1/4} = 0.02960$, i.e. only $54$ intervals — a **104-fold** reduction in table size at the same accuracy.

$$
\boxed{h \le 2.83\times10^{-3} \Rightarrow 557 \text{ entries; cubic spline needs only } 54 \text{ intervals at } 10^{-8}}
$$

*Key takeaway:* Order of accuracy is a memory multiplier — going from $O(h^2)$ to $O(h^4)$ shrinks a $10^{-8}$-accurate table by two orders of magnitude.

### Problem L1.3: The node polynomial, equispaced versus Chebyshev

For $n = 4$ on $[-1,1]$, compute $\lVert \omega_5 \rVert_\infty$ for equispaced nodes and for the roots of $T_5$, and state the general trend for $n = 10, 20$.

**Solution.**

**Equispaced.** Nodes $-1, -\tfrac12, 0, \tfrac12, 1$, so $\omega_5(x) = x(x^2 - \tfrac14)(x^2 - 1)$. Setting $\omega_5' = 0$ and maximizing numerically over $[-1,1]$ gives the peak near $x = \pm 0.8222$ with

$$
\lVert \omega_5 \rVert_\infty = 0.11348\ldots
$$

**Chebyshev roots.** With $x_k = \cos\frac{(2k+1)\pi}{10}$, $\omega_5 = 2^{-4}T_5$, so by equioscillation

$$
\lVert \omega_5 \rVert_\infty = 2^{-4} = 0.0625 \quad \text{exactly,}
$$

attained at five interior points with alternating signs — the minimax value over all monic quintics.

**Trend.** Computed on a fine grid:

| $n$ | equispaced $\Vert \omega_{n+1} \Vert_\infty$ | Chebyshev $= 2^{-n}$ | ratio |
| :--- | :--- | :--- | :--- |
| 4 | $1.135 \times 10^{-1}$ | $6.25\times10^{-2}$ | 1.82 |
| 10 | $8.532\times10^{-3}$ | $9.766\times10^{-4}$ | 8.74 |
| 20 | $2.337\times10^{-4}$ | $9.537\times10^{-7}$ | 245 |

The ratio grows geometrically — this is the node-side origin of Runge's phenomenon.

$$
\boxed{\text{equispaced } 0.11348 \text{ vs Chebyshev } 2^{-4} = 0.0625; \text{ the gap grows like } 2^{n}/(e n \log n)}
$$

*Key takeaway:* Chebyshev nodes are optimal *by construction*: they make the node polynomial equioscillate at the minimax level $2^{1-n}$ for monic degree $n$.

### Problem L1.4: A natural cubic spline by hand

Construct the natural cubic spline through $(0, 0), (1, 1), (2, 0)$. Give both pieces explicitly and verify all continuity conditions.

**Solution.**

Here $n = 2$, $h_0 = h_1 = 1$, and the moment equation for the single interior knot $i = 1$ is

$$
h_0 M_0 + 2(h_0 + h_1)M_1 + h_1 M_2 = 6\left( \frac{y_2 - y_1}{h_1} - \frac{y_1 - y_0}{h_0} \right) = 6\bigl( (-1) - (1) \bigr) = -12 .
$$

Natural conditions give $M_0 = M_2 = 0$, so $4 M_1 = -12$ and $M_1 = -3$.

**Piece on $[0,1]$.** Using the moment formula of Proof 6 with $h_0 = 1$:

$$
s(x) = \frac{M_1 x^3}{6} + \left(y_1 - \frac{M_1}{6}\right)x = -\frac{x^3}{2} + \frac{3}{2}x .
$$

**Piece on $[1,2]$.** By the same formula (or by the symmetry $x \mapsto 2 - x$):

$$
s(x) = -\frac{(2-x)^3}{2} + \frac{3}{2}(2-x) .
$$

**Verification.**
- Interpolation: $s(0) = 0$; $s(1^-) = -\tfrac12 + \tfrac32 = 1$; $s(1^+) = -\tfrac12 + \tfrac32 = 1$; $s(2) = 0$. ✔
- $s'(x) = -\tfrac32 x^2 + \tfrac32$ on the left, $s'(x) = \tfrac32 (2-x)^2 - \tfrac32$ on the right; at $x=1$ both give $0$. ✔
- $s''(x) = -3x$ on the left, $s''(x) = -3(2-x)$ on the right; at $x=1$ both give $-3 = M_1$; at the ends $s''(0) = 0$ and $s''(2) = 0$. ✔

Midpoint value: $s(0.5) = -\tfrac{1}{16} + \tfrac{3}{4} = 0.6875$.

$$
\boxed{s(x) = -\tfrac12 x^3 + \tfrac32 x \text{ on } [0,1], \quad s(x) = -\tfrac12 (2-x)^3 + \tfrac32 (2-x) \text{ on } [1,2]}
$$

*Key takeaway:* Using the second derivatives $M_i$ as unknowns makes $C^2$ continuity automatic and reduces the whole construction to one tridiagonal system.

### Problem L1.5: Cubic Hermite basis functions

Derive the four cubic Hermite basis functions on $[0,1]$ — the unique cubics $h_{00}, h_{10}, h_{01}, h_{11}$ satisfying the value/derivative cardinality conditions at $0$ and $1$ — and state the interpolant on a general interval $[x_i, x_{i+1}]$.

**Solution.**

We want $H(t) = f_0 h_{00}(t) + m_0 h_{10}(t) + f_1 h_{01}(t) + m_1 h_{11}(t)$ with $H(0) = f_0$, $H'(0) = m_0$, $H(1) = f_1$, $H'(1) = m_1$. This forces the cardinality table

$$
h_{00}: (1,0,0,0), \quad h_{10}: (0,1,0,0), \quad h_{01}: (0,0,1,0), \quad h_{11}: (0,0,0,1)
$$

for the data vector $(H(0), H'(0), H(1), H'(1))$.

**Construction.** $h_{00}$ must have a double root at $t = 1$ (value and derivative vanish there), so $h_{00} = (at + b)(t-1)^2$; imposing $h_{00}(0) = 1$ gives $b = 1$, and $h_{00}'(0) = a - 2b = 0$ gives $a = 2$:

$$
h_{00}(t) = (2t+1)(t-1)^2 = 2t^3 - 3t^2 + 1 .
$$

Similarly $h_{10} = t(t-1)^2 = t^3 - 2t^2 + t$; and by the reflection $t \mapsto 1-t$ (which swaps the two ends and flips derivative signs),

$$
h_{01}(t) = -2t^3 + 3t^2, \qquad h_{11}(t) = t^3 - t^2 .
$$

**Sanity checks.** $h_{00} + h_{01} = 1$ (partition of unity, so constants are reproduced) and $h_{10} + h_{11} = t^3 - 2t^2 + t + t^3 - t^2$… more usefully, $0\cdot h_{00} + 1\cdot h_{10} + 1\cdot h_{01} + 1 \cdot h_{11} = t$, reproducing the identity: $t^3 - 2t^2 + t - 2t^3 + 3t^2 + t^3 - t^2 = t$. ✔

**General interval.** With $h = x_{i+1} - x_i$ and $t = (x - x_i)/h$,

$$
H(x) = f_i\,h_{00}(t) + h\,m_i\,h_{10}(t) + f_{i+1}\,h_{01}(t) + h\,m_{i+1}\,h_{11}(t),
$$

the factor $h$ appearing because $\frac{dH}{dx} = \frac{1}{h}\frac{dH}{dt}$. The error is $\frac{f^{(4)}(\xi)}{4!}(x-x_i)^2(x-x_{i+1})^2$, maximized at the midpoint: $\lVert f - H \rVert_\infty \le \frac{h^4}{384}\lVert f^{(4)} \rVert_\infty$.

$$
\boxed{h_{00} = 2t^3 - 3t^2 + 1,\; h_{10} = t^3 - 2t^2 + t,\; h_{01} = -2t^3 + 3t^2,\; h_{11} = t^3 - t^2}
$$

*Key takeaway:* Cubic Hermite interpolation is purely local ($C^1$, two points at a time), $O(h^4)$ accurate, and is the basis of PCHIP, Catmull–Rom splines, and CSS/animation easing curves.

### Problem L1.6: Why the Vandermonde route is a trap

For $n+1$ equispaced nodes on $[0,1]$, explain why solving $Va = y$ for monomial coefficients loses accuracy, quantify the loss for $n = 20$ and $n = 40$ in double precision, and give the two fixes.

**Solution.**

**Mechanism.** The monomial basis $\{1, x, \ldots, x^n\}$ is nearly linearly dependent on $[0,1]$: for large $k$ the functions $x^k$ and $x^{k+1}$ are almost indistinguishable in the sup norm. Nearly dependent columns mean a nearly singular Gram matrix, hence a huge condition number for $V$. The classical asymptotic for equispaced nodes on $[0,1]$ is

$$
\kappa_2(V) \sim \frac{(1+\sqrt2)^{\,n+1}}{\sqrt{\pi n}} \approx 2.414^{\,n} .
$$

**Digit accounting.** A relative perturbation of size $u = 2^{-53} \approx 1.1\times10^{-16}$ in the data produces a relative error up to $\kappa_2(V)\,u$ in $a$.

- $n = 20$: $\kappa_2 \approx 2.414^{21}/\sqrt{20\pi} = 1.4\times10^{7}$, so about $7$ of the $16$ digits are gone.
- $n = 40$: $\kappa_2 \approx 2.414^{41}/\sqrt{40\pi} = 4.4\times10^{14}$ — only about $1.5$ digits survive; the computed coefficients are essentially noise, even though the interpolating polynomial itself is a perfectly well-conditioned function of the data.

Note the crucial distinction: the *problem* (evaluate the interpolant) is well conditioned at Chebyshev nodes; the *representation* (monomial coefficients) is what is ill conditioned. The algorithm, not the mathematics, is at fault.

**Fix 1 — change the basis.** Use the Newton or Chebyshev basis, or the barycentric form which never forms coefficients at all; barycentric evaluation is backward stable for degrees in the thousands.

**Fix 2 — change the nodes.** Chebyshev nodes reduce $\kappa_2(V)$ substantially (though the basis change matters more); the combination of Chebyshev nodes *and* Chebyshev basis gives $\kappa_2 = O(1)$ growth mild enough to be irrelevant in practice.

$$
\boxed{\kappa_2(V) \approx 2.414^{\,n}: \; 1.4\times10^{7} \text{ at } n = 20, \; 4.4\times10^{14} \text{ at } n = 40}
$$

*Key takeaway:* Never compute interpolating-polynomial coefficients by solving a Vandermonde system — the object is fine, the coordinates are not.

## Level 2 — Applications in AI/ML & Physics

### Problem L2.1: A learning-rate schedule as an interpolant

A training run specifies learning rates at milestones: $\mathrm{lr}(0) = 0$, $\mathrm{lr}(1000) = 10^{-3}$ (end of warmup), $\mathrm{lr}(50000) = 10^{-5}$. Compare (a) piecewise-linear interpolation, (b) a natural cubic spline, (c) cosine decay after linear warmup, and explain which is safe to deploy.

**Solution.**

**(a) Piecewise linear.** A degree-1 spline: $\mathrm{lr}(t) = 10^{-6} t$ on $[0,1000]$, then

$$
\mathrm{lr}(t) = 10^{-3} - \frac{9.9\times10^{-4}}{49000}(t - 1000), \quad t \in [1000, 50000].
$$

Monotone, non-negative, trivially safe, but $C^0$ only — the slope discontinuity at $t = 1000$ is harmless for SGD (the schedule is not differentiated).

**(b) Natural cubic spline through the three points.** With $h_0 = 1000$, $h_1 = 49000$ and $M_0 = M_2 = 0$:

$$
2(h_0 + h_1)M_1 = 6\left( \frac{y_2 - y_1}{h_1} - \frac{y_1 - y_0}{h_0} \right) = 6\left( \frac{-9.9\times10^{-4}}{49000} - \frac{10^{-3}}{1000} \right) = 6(-2.0204\times10^{-8} - 10^{-6}),
$$

so $M_1 = \frac{6(-1.0202\times10^{-6})}{100000} = -6.121\times10^{-11}$. Evaluating the resulting spline on the decay interval gives

$$
\max_{t} s(t) = 1.0015\times10^{-2} \ \text{ at } t \approx 21144,
$$

i.e. **ten times the intended peak learning rate**, reached 20000 steps *after* the peak was supposed to be over. The cause is the $49\times$ mismatch in interval lengths: the spline must bend gently over the long interval to reconcile a steep entry slope with $s'' = 0$ at the far end, and it does so by ballooning upward. A run using this schedule diverges immediately.

**(c) Cosine decay after linear warmup** (the standard modern choice):

$$
\mathrm{lr}(t) = \begin{cases} \mathrm{lr}_{\max}\,\dfrac{t}{T_w}, & t \le T_w, \\[2mm] \mathrm{lr}_{\min} + \tfrac12(\mathrm{lr}_{\max} - \mathrm{lr}_{\min})\left(1 + \cos\dfrac{\pi (t - T_w)}{T - T_w}\right), & t \gt T_w. \end{cases}
$$

This is a *shape-constrained* interpolant: it hits all three milestones, is monotone decreasing after warmup, is $C^1$ at the peak (both one-sided derivatives of the cosine branch vanish at $t = T_w$ only in the derivative of the cosine piece — the kink at $T_w$ is deliberate), and is bounded in $[\mathrm{lr}_{\min}, \mathrm{lr}_{\max}]$ by construction.

$$
\boxed{\text{Cubic spline peaks at } 1.0\times10^{-2} = 10\times \mathrm{lr}_{\max}; \text{ use shape-constrained schedules}}
$$

*Key takeaway:* A schedule is an interpolation problem with a hard **range constraint** and a **monotonicity constraint** — exactly the situation where PCHIP-style or analytic (cosine) interpolants beat smooth splines, especially on wildly non-uniform milestone spacing.

### Problem L2.2: Spline overshoot in probability calibration

A classifier's reliability diagram gives binned (confidence, accuracy) pairs $(0,0), (1,0), (2,0), (3,1), (4,1), (5,1)$ on a rescaled axis. Fit a natural cubic spline and a PCHIP interpolant, quantify the overshoot, and explain the consequence for calibrated probabilities.

**Solution.**

**Natural cubic spline.** Solving the $4\times4$ tridiagonal moment system ($h_i = 1$ throughout, interior knots $i = 1,\ldots,4$) and evaluating on a fine grid gives

$$
\min_{x \in [0,5]} s(x) = -0.10924 \ \text{(at } x \approx 1.616\text{)}, \qquad \max_{x\in[0,5]} s(x) = 1.10924 \ \text{(at } x \approx 3.384\text{)},
$$

with $s(2.5) = 0.5$ exactly by the antisymmetry of the data about the centre. The overshoot is $10.9\%$ on **both** sides.

**PCHIP.** The Fritsch–Carlson slope limiter sets $m_i = 0$ whenever consecutive secant slopes have opposite signs or one vanishes, and otherwise caps $\lvert m_i \rvert \le 3 \min(\lvert \Delta_{i-1}\rvert, \lvert \Delta_i \rvert)$. Here the data are flat–flat–jump–flat–flat, so most slopes are clamped to $0$ and the interpolant satisfies

$$
\min_{x} p(x) = 0, \qquad \max_x p(x) = 1 ,
$$

monotone nondecreasing throughout.

**Why it matters.** A calibration map $g$ converts raw confidence into a probability. The spline returns $g = -0.109$ and $g = 1.109$ — outside $[0,1]$. Downstream that yields negative log-likelihood of a negative number ($\mathrm{NaN}$), invalid Brier scores, and non-monotone rankings (a *less* confident prediction mapped to a *higher* probability), which destroys AUC-preserving guarantees.

**Standard practice.** Use **isotonic regression** (the exact monotone least-squares projection, computed by pool-adjacent-violators in $O(n)$), Platt scaling (a monotone logistic fit), or a monotone spline. The $C^2$ smoothness of the cubic spline is worth nothing here; the monotonicity is everything.

$$
\boxed{\text{cubic spline: } [-0.109,\, 1.109]; \quad \text{PCHIP: } [0,\,1] \text{ and monotone}}
$$

*Key takeaway:* When the output has a physical range or ordering, shape preservation dominates smoothness — this is the single most common interpolation bug in ML pipelines.

### Problem L2.3: Resizing positional embeddings for a Vision Transformer

A ViT pretrained at $224 \times 224$ with patch size $16$ has a $14\times14$ grid of learned positional embeddings, each a $768$-vector. Fine-tuning at $384\times384$ needs a $24 \times 24$ grid. Explain the interpolation performed, its cost, and why bicubic rather than nearest or bilinear.

**Solution.**

**The operation.** Reshape the $196 \times 768$ embedding table to a tensor of shape $(768, 14, 14)$ — treat each of the $768$ channels as a scalar function sampled on a $14\times14$ grid — and resample it onto a $24\times24$ grid. This is exactly two-dimensional interpolation applied channel-wise:

```python
import torch, torch.nn.functional as Fn
pe = pos_embed[:, 1:, :]                       # drop CLS token: (1, 196, 768)
pe = pe.reshape(1, 14, 14, 768).permute(0, 3, 1, 2)   # (1, 768, 14, 14)
pe = Fn.interpolate(pe, size=(24, 24), mode='bicubic', align_corners=False)
pe = pe.permute(0, 2, 3, 1).reshape(1, 576, 768)
```

**Tensor-product structure.** A bicubic resample is a tensor product of 1-D cubic interpolations: interpolate along rows, then along columns. Each output value is a weighted sum of the $4 \times 4 = 16$ nearest inputs, so the cost is $O(16 \cdot 576 \cdot 768) \approx 7.1$ million multiply–adds — negligible, and done once at load time.

**Why bicubic.**
- **Nearest neighbour** is a degree-0 spline: $O(h)$ accurate and discontinuous. Discontinuities in positional embeddings inject high-frequency artifacts that the attention layers must unlearn.
- **Bilinear** is a degree-1 spline: $O(h^2)$, continuous but with slope discontinuities at every grid line; the derivative field (which attention effectively probes through relative-position differences) is piecewise constant.
- **Bicubic** (Keys' cubic convolution, a $C^1$ piecewise cubic) is $O(h^3)$–$O(h^4)$ accurate and has a continuous first derivative, so the smooth low-frequency structure of the learned embedding — which is what encodes relative geometry — is preserved.

**The coarse-grid caveat.** With only $14$ samples per axis, the interpolation error is dominated by whether the underlying embedding field is smooth, not by the order of the method. Empirically the smooth Fourier-like structure of learned 2-D positional embeddings is precisely why bicubic resizing works at all and why the fine-tuned model recovers accuracy within a few hundred steps.

$$
\boxed{\text{Bicubic tensor-product spline: } 16 \text{ taps per output, } C^1, O(h^4) \text{ on smooth fields}}
$$

*Key takeaway:* Resolution transfer in vision models is a spline problem, and the smoothness order of the interpolant directly controls how much retraining is needed to repair the transplant.

### Problem L2.4: The smoothing spline and its two limits

The smoothing spline minimizes $J_\lambda[g] = \sum_{i=1}^{N}\bigl(y_i - g(x_i)\bigr)^2 + \lambda \int_a^b \bigl(g''(x)\bigr)^2 dx$ over $g \in C^2$. Identify the minimizer's structure and analyse the limits $\lambda \to 0^+$ and $\lambda \to \infty$. Connect to ridge regression and to Gaussian processes.

**Solution.**

**Structure of the minimizer.** For any candidate $g$, replace it by the natural cubic spline $s$ interpolating the values $g(x_i)$. The data-fit term is unchanged, and by Theorem 9 (minimum bending energy) $\int (s'')^2 \le \int (g'')^2$. Hence $J_\lambda[s] \le J_\lambda[g]$: **the minimizer is a natural cubic spline with knots at the data points**, regardless of $\lambda$. The infinite-dimensional variational problem collapses to an $N$-dimensional one.

Writing $\mathbf{g} = (g(x_1),\ldots,g(x_N))$ and using the fact that $\int (s'')^2 = \mathbf{g}^{\mathsf T} K \mathbf{g}$ for an explicit positive-semidefinite roughness matrix $K$ (rank $N - 2$, null space = the affine functions), the problem becomes ridge regression in a rotated basis:

$$
\hat{\mathbf g} = \arg\min_{\mathbf g} \lVert \mathbf y - \mathbf g \rVert_2^2 + \lambda\, \mathbf g^{\mathsf T} K \mathbf g \quad \Longrightarrow \quad \hat{\mathbf g} = (I + \lambda K)^{-1}\mathbf y = S_\lambda\, \mathbf y .
$$

**Limit $\lambda \to 0^+$.** The penalty vanishes and $S_\lambda \to I$: the fit becomes the **interpolating** natural cubic spline, $\hat g(x_i) = y_i$. Zero bias, maximal variance — with noisy data this reproduces the noise exactly.

**Limit $\lambda \to \infty$.** The penalty dominates, forcing $\mathbf g^{\mathsf T} K \mathbf g \to 0$, i.e. $g'' \equiv 0$, i.e. $g$ affine. Subject to that, the data term is minimized by ordinary least squares: $\hat g \to$ the **least-squares straight line**. Maximal bias, minimal variance.

**Effective degrees of freedom.** $\mathrm{df}(\lambda) = \operatorname{tr} S_\lambda = \sum_{k} \frac{1}{1 + \lambda d_k}$ with $d_k$ the eigenvalues of $K$, decreasing smoothly from $N$ (at $\lambda = 0$) to $2$ (as $\lambda \to \infty$). This is exactly the ridge-regression shrinkage profile $\frac{\sigma_k^2}{\sigma_k^2 + \lambda}$ and is what generalized cross-validation optimizes.

**Gaussian-process identity.** $S_\lambda = (I + \lambda K)^{-1}$ is the posterior-mean operator of a GP with prior $g''(x) = \sigma\,dW(x)$ (twice-integrated Brownian motion) and observation noise $\lambda^{-1}$-scaled. So the smoothing spline *is* GP regression with a specific kernel, and $\lambda$ is the noise-to-signal ratio.

$$
\boxed{\hat g = (I + \lambda K)^{-1} y: \quad \lambda \to 0 \Rightarrow \text{interpolating spline}, \quad \lambda \to \infty \Rightarrow \text{least-squares line}}
$$

*Key takeaway:* Interpolation and linear regression are the two endpoints of one continuous family; $\lambda$ is the bias–variance dial, and the minimum-energy theorem is what makes the family finite-dimensional.

### Problem L2.5: Parameter budget of a KAN spline layer

A Kolmogorov–Arnold Network replaces each weight by a learnable univariate B-spline. For a layer with $n_{\mathrm{in}} = 512$, $n_{\mathrm{out}} = 512$, cubic B-splines ($k = 3$) on a grid with $G = 5$ intervals, compute the parameter count against an MLP layer, the number of basis functions active per input, and the cost of grid refinement.

**Solution.**

**Basis size.** A spline of degree $k$ on a grid with $G$ intervals (hence $G + 2k + 1$ knots including the padding) has

$$
G + k = 5 + 3 = 8
$$

B-spline basis functions. Each edge $(i \to j)$ carries a function $\phi_{ij}(x) = w_b\,b(x) + w_s \sum_{m=1}^{8} c^{(ij)}_m B_m(x)$.

**Parameter count.**

$$
P_{\mathrm{KAN}} = n_{\mathrm{in}} n_{\mathrm{out}} (G + k) = 512 \times 512 \times 8 = 2{,}097{,}152,
$$

plus $2 n_{\mathrm{in}} n_{\mathrm{out}} = 524{,}288$ scale parameters if $w_b, w_s$ are per edge, versus

$$
P_{\mathrm{MLP}} = n_{\mathrm{in}} n_{\mathrm{out}} + n_{\mathrm{out}} = 262{,}144 + 512 = 262{,}656 .
$$

The KAN layer is $\approx 8\times$ (or $10\times$ with scales) larger for the same width. Its claimed advantage is *expressivity per parameter on structured/compositional functions*, not raw parameter efficiency at fixed width.

**Locality.** A degree-$k$ B-spline has support on $k+1 = 4$ consecutive intervals, so for any input value $x$ only $4$ of the $8$ coefficients per edge are active:

$$
\phi(x) = \sum_{m = \mu-3}^{\mu} c_m B_m(x), \quad x \in [t_\mu, t_{\mu+1}).
$$

Gradients are therefore **sparse**: each forward sample updates only $4$ coefficients per edge, which is both an efficiency win and a source of the "dead grid region" problem when the input distribution shifts away from the grid range (hence the grid-update heuristics in KAN implementations).

**Grid refinement.** Doubling to $G = 10$ raises the basis to $13$ functions per edge, i.e. $512^2 \times 13 = 3{,}407{,}872$ parameters. Crucially, knot insertion is **exact**: the coarse spline lies in the fine spline space, so the refined coefficients can be computed by the Oslo/Boehm algorithm with *no loss of function value* — a rare case where a network can be grown without perturbing its current function. That property comes directly from the nestedness of spline spaces under knot refinement.

$$
\boxed{P_{\mathrm{KAN}} = n_{\mathrm{in}} n_{\mathrm{out}} (G + k) = 2{,}097{,}152 \approx 8 \times P_{\mathrm{MLP}}; \; k+1 = 4 \text{ active bases per input}}
$$

*Key takeaway:* Local support gives B-spline layers sparse gradients and exact, function-preserving capacity growth by knot insertion — the two properties that classical dense layers lack.

### Problem L2.6: Sizing an equation-of-state table for a simulation

A hydrodynamics code must evaluate a smooth thermodynamic function $f(T)$ millions of times per timestep on $T \in [0, 1]$ (normalized), to relative accuracy $10^{-8}$, with $\lVert f^{(4)} \rVert_\infty \approx 1$ and $\lVert f'' \rVert_\infty \approx 1$. Compare table sizes for linear and cubic-spline interpolation, and discuss the accuracy of the *derivative*.

**Solution.**

**Linear interpolation.** $\frac{h^2}{8}\lVert f'' \rVert_\infty \le 10^{-8}$ gives

$$
h \le \sqrt{8\times10^{-8}} = 2.8284\times10^{-4} \implies N = \lceil 1/h \rceil = 3536 \text{ intervals}.
$$

**Cubic spline (clamped).** $\frac{5}{384}h^4 \lVert f^{(4)}\rVert_\infty \le 10^{-8}$ gives

$$
h^4 \le \frac{384\times10^{-8}}{5} = 7.68\times10^{-7} \implies h \le 0.029603 \implies N = 34 \text{ intervals}.
$$

A **104-fold** reduction in table entries. At $8$ bytes per entry and $10^3$ tables (one per species/opacity channel), that is $28$ MB versus $272$ kB — the difference between thrashing the L2 cache on every lookup and living comfortably in it. For a memory-bandwidth-bound kernel this is the whole performance story.

**Derivative accuracy — the sting.** Differentiating an interpolant loses one order:

$$
\lVert f' - s' \rVert_\infty \le \frac{h^3}{24}\lVert f^{(4)} \rVert_\infty .
$$

At $h = 0.0296$: $\frac{(0.0296)^3}{24} = 1.08\times10^{-6}$ — only about $10^{-6}$, not $10^{-8}$. If the code needs $\partial f/\partial T$ (e.g. sound speed, or a Jacobian for an implicit solver) to $10^{-8}$, solve $\frac{h^3}{24} \le 10^{-8}$:

$$
h \le (2.4\times10^{-7})^{1/3} = 6.214\times10^{-3} \implies N = 161 \text{ intervals}.
$$

Still $22\times$ smaller than the linear table, but $4.7\times$ larger than the value-only requirement.

**Second derivative.** $\lVert f'' - s'' \rVert_\infty = O(h^2)$, and at the ends of a *natural* spline it is exactly zero — usually wrong. If second derivatives matter, use clamped or not-a-knot conditions and a quintic spline.

$$
\boxed{N_{\text{linear}} = 3536 \text{ vs } N_{\text{spline}} = 34 \text{ for values}; \; N_{\text{spline}} = 161 \text{ if } f' \text{ also needs } 10^{-8}}
$$

*Key takeaway:* Table sizing is an error-bound inversion, and each derivative you need costs one order of accuracy — size the table for the highest derivative the code consumes.

## Level 3 — Challenge

### Problem L3.1: Lebesgue constants and near-best approximation

Prove that for any node set, $\lVert f - \Pi_n f \rVert_\infty \le (1 + \Lambda_n)\,\mathrm{dist}_\infty(f, \mathbb{P}_n)$ where $\Lambda_n = \lVert \Pi_n \rVert_\infty$. Then use the known asymptotics to compare equispaced and Chebyshev nodes at $n = 20$.

**Solution.**

**Step 1 — $\Lambda_n$ is the operator norm.** $\Pi_n f = \sum_i f(x_i) L_i$, so

$$
\lvert (\Pi_n f)(x) \rvert \le \lVert f \rVert_\infty \sum_i \lvert L_i(x) \rvert \le \lVert f \rVert_\infty \max_x \sum_i \lvert L_i(x) \rvert = \Lambda_n \lVert f \rVert_\infty .
$$

The bound is attained (choose a continuous $f$ with $\lVert f \rVert_\infty = 1$ and $f(x_i) = \operatorname{sign} L_i(x^\star)$ at the maximizing $x^\star$), so $\lVert \Pi_n \rVert_\infty = \Lambda_n$ exactly.

**Step 2 — the near-best inequality.** Let $p^\star \in \mathbb{P}_n$ be a best approximation, $\lVert f - p^\star \rVert_\infty = \mathrm{dist}_\infty(f, \mathbb{P}_n) =: E_n(f)$. Since $\Pi_n$ is a projection, $\Pi_n p^\star = p^\star$. Then

$$
f - \Pi_n f = (f - p^\star) + (p^\star - \Pi_n f) = (f - p^\star) - \Pi_n(f - p^\star),
$$

so by the triangle inequality and Step 1,

$$
\lVert f - \Pi_n f \rVert_\infty \le \lVert f - p^\star \rVert_\infty + \Lambda_n \lVert f - p^\star \rVert_\infty = (1 + \Lambda_n) E_n(f). \qquad \blacksquare
$$

**Step 3 — the comparison at $n = 20$.**

- **Equispaced**: $\Lambda_n \sim \dfrac{2^{\,n+1}}{e\,n\log n}$. At $n = 20$ this asymptotic gives $\frac{2^{21}}{2.718 \times 20 \times 3.00} \approx 1.29\times10^{4}$, and direct computation of $\max_x \sum_i \lvert L_i(x)\rvert$ on a fine grid gives $\Lambda_{20} = 1.099\times10^{4}$. So the interpolant may be $10^4$ times worse than best approximation — and at $n = 60$ the factor exceeds $10^{14}$, past machine precision.
- **Chebyshev**: $\Lambda_n \sim \dfrac{2}{\pi}\log n + 1$. At $n = 20$ the asymptotic gives $2.91$ and the computed value is $2.87$. The interpolant is within a factor of about $4$ of the *best possible* polynomial of that degree, for every $f$, forever — the growth is logarithmic, so even $n = 10^{6}$ gives $\Lambda_n \approx 9.8$.

**Why this is the definitive argument for Chebyshev nodes.** Jackson's theorems bound $E_n(f)$ (e.g. $E_n(f) \le C n^{-k}\lVert f^{(k)} \rVert$ for $f \in C^k$, and $E_n(f) = O(\rho^{-n})$ for $f$ analytic in a Bernstein $\rho$-ellipse). Multiplying by $\Lambda_n$: Chebyshev interpolation inherits essentially the full Jackson rate, degraded only by $\log n$; equispaced interpolation multiplies it by $2^{n}$ and typically diverges.

$$
\boxed{\lVert f - \Pi_n f \rVert_\infty \le (1 + \Lambda_n) E_n(f); \quad \Lambda_{20}^{\text{equi}} = 1.10\times10^{4}, \; \Lambda_{20}^{\text{Cheb}} = 2.87}
$$

*Key takeaway:* The Lebesgue constant is the single number separating a usable node set from a catastrophic one, and it is a property of the nodes alone — independent of $f$.

### Problem L3.2: Deriving the barycentric formula

Starting from the Lagrange form, derive the second (true) barycentric formula, prove the weights for Chebyshev points of the second kind are $w_i = (-1)^i \delta_i$ with $\delta_0 = \delta_n = \tfrac12$, and explain why the formula is numerically stable while the classical Lagrange form is slow.

**Solution.**

**Step 1 — factor out the node polynomial.** Let $\ell(x) = \prod_{j=0}^{n}(x - x_j)$ and $w_i = \dfrac{1}{\prod_{j\neq i}(x_i - x_j)}$. Then

$$
L_i(x) = \prod_{j\neq i}\frac{x - x_j}{x_i - x_j} = \frac{\ell(x)}{x - x_i}\,w_i ,
$$

because $\ell(x)/(x-x_i) = \prod_{j\neq i}(x - x_j)$. Hence the **first barycentric form**

$$
p_n(x) = \ell(x) \sum_{i=0}^{n} \frac{w_i}{x - x_i}\, y_i .
$$

This already costs only $O(n)$ per evaluation once the $w_i$ are precomputed in $O(n^2)$.

**Step 2 — eliminate $\ell(x)$.** Apply the identity to the constant function $f \equiv 1$, whose interpolant is exactly $1$ by uniqueness:

$$
1 = \ell(x)\sum_{i=0}^{n}\frac{w_i}{x - x_i} .
$$

Dividing the first form by this identity removes $\ell$ entirely:

$$
\boxed{\;p_n(x) = \frac{\displaystyle\sum_{i=0}^n \frac{w_i}{x - x_i} y_i}{\displaystyle\sum_{i=0}^n \frac{w_i}{x - x_i}}\;}
$$

the **second barycentric formula**. It is manifestly scale-invariant in $w$: multiplying all $w_i$ by a constant leaves $p_n$ unchanged, which is what permits the clean closed forms below.

**Step 3 — Chebyshev weights of the second kind.** With $x_i = \cos(i\pi/n)$, the nodes are the extrema of $T_n$, i.e. the roots of $(1-x^2)U_{n-1}(x)$. A direct evaluation of $\prod_{j\neq i}(x_i - x_j)$ using $\sin$-product identities gives $\prod_{j\neq i}(x_i - x_j) = \frac{(-1)^i n}{2^{\,n-1}\delta_i}$ (with $\delta_0 = \delta_n = \tfrac12$, $\delta_i = 1$ otherwise), so

$$
w_i = \frac{2^{\,n-1}}{n}(-1)^i \delta_i \;\;\propto\;\; (-1)^i \delta_i ,
$$

and by scale invariance the constants can be dropped. **No computation is needed at all** — the weights are $+1, -1, +1, \ldots$ with halves at the ends.

**Step 4 — why it is stable.** Higham (2004) proved the second barycentric formula is **backward stable** for node sets with small Lebesgue constant: the computed value is the exact interpolant of slightly perturbed data. The potentially catastrophic quantities $w_i$ can span many orders of magnitude for bad node sets, but for Chebyshev points they are all $\pm 1$, and the division by $\sum_i w_i/(x-x_i)$ cancels the common growth. The dangerous-looking division by $x - x_i$ is handled by the standard "exact hit" branch (if $x$ equals a node, return $y_i$); near-hits are harmless because the same large term dominates numerator and denominator.

**Cost.** Classical Lagrange: $O(n^2)$ per evaluation point. Barycentric: $O(n^2)$ once for the weights (free for Chebyshev), then $O(n)$ per point. For $m$ evaluation points the totals are $O(mn^2)$ versus $O(n^2 + mn)$.

*Key takeaway:* The barycentric formula makes Lagrange interpolation both the fastest and the most stable representation — the historical prejudice against it applies only to the naive $O(n^2)$ evaluation.

### Problem L3.3: The Hermite interpolation error formula

Prove that if $H \in \mathbb{P}_{2n+1}$ matches $f$ and $f'$ at distinct nodes $x_0,\ldots,x_n$ and $f \in C^{2n+2}[a,b]$, then for each $x$ there is $\xi_x$ with

$$
f(x) - H(x) = \frac{f^{(2n+2)}(\xi_x)}{(2n+2)!}\prod_{i=0}^{n}(x - x_i)^2 .
$$

**Solution.**

**Setup.** Fix $x \notin \{x_i\}$ (at a node both sides vanish). Set $\Omega(t) = \prod_{i=0}^{n}(t - x_i)^2$, a **monic** polynomial of degree $2n+2$, and define the constant

$$
K = \frac{f(x) - H(x)}{\Omega(x)}, \qquad g(t) = f(t) - H(t) - K\,\Omega(t) .
$$

**Counting zeros with multiplicity.** At each node $x_i$:

$$
g(x_i) = f(x_i) - H(x_i) - K\cdot 0 = 0, \qquad g'(x_i) = f'(x_i) - H'(x_i) - K\,\Omega'(x_i) = 0,
$$

using $H(x_i) = f(x_i)$, $H'(x_i) = f'(x_i)$, and $\Omega'(x_i) = 0$ (each factor is squared, so $x_i$ is a double root of $\Omega$). Hence each of the $n+1$ nodes is a **double** zero of $g$, and $g(x) = 0$ as well: counted with multiplicity, $g$ has $2(n+1) + 1 = 2n + 3$ zeros in $[a,b]$.

**Repeated Rolle.** Between two consecutive *distinct* zeros of $g$ lies a zero of $g'$; moreover each double zero $x_i$ of $g$ is itself a zero of $g'$. Ordering the $n+2$ distinct zeros of $g$ (the nodes plus $x$), Rolle supplies $n+1$ new zeros of $g'$ strictly between them, and the $n+1$ nodes contribute $n+1$ more, for a total of at least $2n+2$ distinct zeros of $g'$. Applying ordinary Rolle repeatedly, $g''$ has $\ge 2n+1$ zeros, …, and $g^{(2n+2)}$ has at least one zero $\xi_x$ in the open interval spanned by $x$ and the nodes.

**Differentiate $2n+2$ times.** $H \in \mathbb{P}_{2n+1}$ so $H^{(2n+2)} \equiv 0$; $\Omega$ is monic of degree $2n+2$ so $\Omega^{(2n+2)} \equiv (2n+2)!$. Therefore

$$
0 = g^{(2n+2)}(\xi_x) = f^{(2n+2)}(\xi_x) - K\,(2n+2)! \implies K = \frac{f^{(2n+2)}(\xi_x)}{(2n+2)!},
$$

and substituting into the definition of $K$ gives the formula. $\blacksquare$

**Consequences.**
- **Sign.** $\prod (x-x_i)^2 \ge 0$, so on any interval where $f^{(2n+2)}$ has constant sign, $H$ lies entirely on one side of $f$ — unlike ordinary interpolation, which weaves across.
- **Order.** For a single interval with $n = 1$ (cubic Hermite, $2n+2 = 4$): $\lvert f - H \rvert \le \frac{\lVert f^{(4)}\rVert_\infty}{24}\max (x-x_0)^2(x-x_1)^2 = \frac{h^4}{384}\lVert f^{(4)}\rVert_\infty$, since the max of $(x-x_0)^2(x-x_1)^2$ over the interval is $(h^2/4)^2 = h^4/16$ and $24 \times 16 = 384$.
- **Confluent divided differences.** The same result follows from the general formula $f - p = f[x_0,\ldots,x_m, x]\,\omega(x)$ with each node repeated twice, using $f[x_i, x_i] = f'(x_i)$.

*Key takeaway:* Matching derivatives doubles the effective node count: $n+1$ nodes with values *and* slopes give the error of $2n+2$ ordinary nodes, and the squared node polynomial makes the error one-signed.

### Problem L3.4: The spline system is always solvable — and stably

Prove that the cubic-spline moment matrix is nonsingular for any strictly increasing knots, derive the resulting bound on the moments, and show the Thomas algorithm needs no pivoting. Then explain the geometric decay of the influence of one data point.

**Solution.**

**Step 1 — the matrix.** For natural end conditions the unknowns are $M_1, \ldots, M_{n-1}$ and row $i$ reads

$$
h_{i-1} M_{i-1} + 2(h_{i-1} + h_i)M_i + h_i M_{i+1} = d_i, \qquad d_i = 6\bigl(f[x_i, x_{i+1}] - f[x_{i-1}, x_i]\bigr),
$$

so $A$ is tridiagonal with $a_{ii} = 2(h_{i-1} + h_i)$, sub/super-diagonals $h_{i-1}, h_i \gt 0$.

**Step 2 — strict diagonal dominance.** For every row,

$$
\lvert a_{ii} \rvert = 2(h_{i-1} + h_i) \gt h_{i-1} + h_i \ge \sum_{j \neq i} \lvert a_{ij} \rvert ,
$$

with the last inequality an equality for interior rows and strict for the first and last rows (where one off-diagonal is absorbed by the boundary condition). So $A$ is **strictly diagonally dominant by rows**.

**Step 3 — nonsingularity (Levy–Desplanques).** Suppose $A M = 0$ with $M \neq 0$, and pick $k$ with $\lvert M_k \rvert = \max_i \lvert M_i \rvert \gt 0$. Row $k$ gives $a_{kk}M_k = -\sum_{j\neq k} a_{kj}M_j$, hence

$$
\lvert a_{kk} \rvert \,\lvert M_k \rvert \le \sum_{j\neq k}\lvert a_{kj}\rvert\,\lvert M_j \rvert \le \lvert M_k \rvert \sum_{j\neq k}\lvert a_{kj}\rvert \lt \lvert a_{kk}\rvert\,\lvert M_k\rvert,
$$

a contradiction. So $A$ is nonsingular and the spline exists and is unique. (Equivalently: Gershgorin discs centred at $2(h_{i-1}+h_i) \gt 0$ with radius $h_{i-1}+h_i$ exclude the origin, so all eigenvalues are positive — $A$ is in fact symmetric positive definite after the standard symmetric scaling.)

**Step 4 — a bound on the moments.** The same maximum-row argument applied to $AM = d$ gives

$$
\lvert a_{kk}\rvert \lvert M_k \rvert - \sum_{j \neq k}\lvert a_{kj}\rvert \lvert M_k \rvert \le \lvert d_k \rvert \implies \lVert M \rVert_\infty \le \frac{\lVert d \rVert_\infty}{\min_i \bigl( \lvert a_{ii}\rvert - \sum_{j\neq i}\lvert a_{ij}\rvert \bigr)} = \frac{\lVert d \rVert_\infty}{\min_i (h_{i-1} + h_i)} .
$$

On a uniform mesh this is $\lVert M \rVert_\infty \le \frac{\lVert d \rVert_\infty}{2h} \le \frac{3}{h}\max_i \lvert f[x_i,x_{i+1}] - f[x_{i-1},x_i] \rvert \le 3\lVert f'' \rVert_\infty$ — bounded independently of $h$, which is the crux of the $O(h^4)$ convergence proof.

**Step 5 — no pivoting needed.** LU factorization of a strictly diagonally dominant matrix preserves strict diagonal dominance in the Schur complement (a standard induction: after eliminating row 1, $\tilde a_{ij} = a_{ij} - a_{i1}a_{1j}/a_{11}$, and dominance is inherited). Hence no pivot is ever zero or small, growth factors stay $\le 2$, and the Thomas algorithm — $O(n)$ time, $O(n)$ memory, no row interchanges — is backward stable.

**Step 6 — geometric decay of influence.** On a uniform mesh the system is $M_{i-1} + 4M_i + M_{i+1} = d_i$. The homogeneous recursion $r^2 + 4r + 1 = 0$ has roots $r = -2 \pm \sqrt3$, so $A^{-1}$ has entries decaying like

$$
\lvert (A^{-1})_{ij} \rvert \sim C\,(2 - \sqrt3)^{\,\lvert i - j \rvert} \approx C\,(0.2679)^{\lvert i - j\rvert}.
$$

Changing a single $y_j$ therefore perturbs the spline visibly only within a few knots — each knot away damps the effect by $\approx 3.7\times$, so after $6$ knots the influence is below $10^{-3}$.

$$
\boxed{A \text{ strictly diagonally dominant} \Rightarrow \text{unique spline, } O(n) \text{ pivot-free solve, influence decay } (2-\sqrt3)^{\lvert i-j\rvert}}
$$

*Key takeaway:* Diagonal dominance simultaneously delivers existence, uniqueness, an $h$-independent stability bound, pivot-free $O(n)$ solvability, and effective locality — one structural property doing five jobs.